# שלב 10 — סיכום הממצאים ומסקנות

ה‑notebook הזה מרכז את התוצאות מכל שלבי הצינור (01–09) למקום אחד, ומסכם את המסקנות. הוא קורא את הפלטים שנשמרו בכל שלב, אז יש להריץ קודם את השלבים הקודמים.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas

In [ ]:
from pathlib import Path
import json
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
OUT = ROOT / "public_transport_network_notebooks" / "outputs"
print("OUT:", OUT)

## מבנה הרשת (שלב 03)

In [ ]:
summary = json.loads((OUT / "03_network_descriptive_analysis" / "network_summary.json").read_text(encoding="utf-8"))
for k in ["num_nodes", "num_edges_undirected", "avg_degree", "largest_component_share",
          "num_articulation_points", "num_bridges"]:
    print(f"  {k}: {summary[k]}")

## התחנות המרכזיות ביותר (שלב 04)

חמש התחנות עם ה‑Betweenness הגבוה ביותר — תחנות המעבר שהרשת תלויה בהן.

In [ ]:
top_btw = pd.read_csv(OUT / "04_centrality_analysis" / "top_betweenness.csv", encoding="utf-8-sig")
top_btw[["stop_name", "region", "degree", "betweenness", "pagerank"]].head(5)

## עמידות הרשת (שלב 05)

גודל הרכיב הגדול אחרי הסרת 3,000 תחנות לפי כל אסטרטגיה — כמה הרשת קרסה.

In [ ]:
dis = pd.read_csv(OUT / "05_robustness_and_disruption_analysis" / "disruption_results.csv", encoding="utf-8-sig")
final = dis.sort_values("removed").groupby("strategy").tail(1)[["strategy", "removed", "lcc_share"]]
final.sort_values("lcc_share")

## השוואה אזורית (שלב 06) וקהילות (שלב 07)

In [ ]:
reg = pd.read_csv(OUT / "06_regional_comparison" / "regional_summary.csv", encoding="utf-8-sig")
print(reg[["region", "total_stops", "pct_critical", "pct_ap"]].to_string(index=False))
comm = pd.read_csv(OUT / "07_community_detection" / "community_summary.csv", encoding="utf-8-sig")
print(f"\nמספר קהילות Louvain: {len(comm)} | קהילה גדולה ביותר: {comm['size'].max()} תחנות")

## חיזוי קשתות (שלב 09)

דיוק שיטות ה‑Link Prediction לפי AUC.

In [ ]:
lp = pd.read_csv(OUT / "09_link_prediction_and_network_improvement" / "link_prediction_results.csv", encoding="utf-8-sig")
lp.sort_values("auc", ascending=False)

## מסקנות עיקריות

1. **הרשת מחוברת ברובה אבל תלויה במאות נקודות תורפה.** 99.3% מהתחנות ברכיב קשור אחד, אבל יש 900 Articulation Points ו‑971 Bridges שהסרתם מפצלת אזורים.

2. **הסרה ממוקדת הרבה יותר הרסנית מאקראית.** הסרה לפי Degree או Betweenness מפילה את הרכיב הגדול לכ‑30% מגודלו, בעוד הסרה אקראית של אותו מספר תחנות משאירה אותו מעל 85%. הרשת פגיעה להתקפה ממוקדת.

3. **הפריפריה חשופה יותר.** בצפון ובדרום שיעור התחנות הקריטיות גבוה יותר (14% ו‑12.9%) מאשר במרכז (10.8%).

4. **קהילות הרשת תואמות גיאוגרפיה.** Louvain מצא עשרות קהילות מרחביות רציפות, שתואמות אזורי תחבורה טבעיים.

5. **קריטיות מבנית קשה לנבא מ‑embeddings מקומיים** (F1 נמוך), אבל Node2Vec מצוין דווקא ל‑Link Prediction (AUC≈0.99) ולמציאת תחנות דומות.

## עבודה שנותרה / כיווני המשך

- **ניתוח דינמי:** עמידות לפי שעות שיא מול שעות שפל, במקום גרף סטטי אחד.
- **שילוב ביקוש נוסעים:** שקלול המרכזיות במספר העולים/יורדים בפועל, לא רק בטופולוגיה.
- **הסרת מקטעים (קשתות) ולא רק תחנות**, לבחינת תרחישי שיבוש נוספים.
- **בדיקת החיבורים המוצעים** מול שיקולי תכנון אמיתיים בשטח.